# Initial testing of the enrichment functionality. This is deprecated now and won't run. The results were that naive enrichment via regex rules do not actually improve embedding quality when using Nomic embeddings. 

In [1]:
from pathlib import Path
import sys

# annoying boilerplate that adds ../paper-clustering/ to the $PYTHONPATH variable 
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path)

from tqdm import tqdm
from supabase import create_client, Client
import pandas as pd
from pprint import pp

['/home/alfred/git/python/paper-clustering', '/home/alfred/miniconda3/envs/paper-clustering/lib/python311.zip', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.11', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.11/lib-dynload', '', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.11/site-packages']


In [ ]:
CATEGORIES = ["cs.DS", "cs.IT", "cs.CC", "math.CO"]
DAYS_BACK = 10 * 365
from paper_clustering.pipeline import fetch_arxiv_data
from paper_clustering.logger import IngestionLogger
import logging
logging.getLogger().setLevel(logging.INFO)
logger = IngestionLogger("../failure.jsonl", "../skipped_papers.txt")

papers = []
ALL_KEYWORDS = []
def collect_locality_papers(papers, ALL_KEYWORDS):
    # Collect some examples on locality in coding theory
    KEYWORDS = [
        "locally decodable",
        "locally correctable",
        "locally testable",
        "private information retrieval",
        "smooth code",
        "locally recoverable",
    ]
    ALL_KEYWORDS += KEYWORDS
    locality_papers = fetch_arxiv_data(CATEGORIES, 150, logger, DAYS_BACK, KEYWORDS)
    papers += locality_papers
    
def collect_coding_papers(papers, ALL_KEYWORDS):
    # Collect some examples on general coding theory
    KEYWORDS = [
        "erasure coding",
        "list decoding",
        "list recovery",
        "random code",
        "matching vector"
    ]
    ALL_KEYWORDS += KEYWORDS
    coding_papers = fetch_arxiv_data(CATEGORIES, 75, logger, DAYS_BACK, KEYWORDS)
    papers += coding_papers

def collect_itcr_papers(papers, ALL_KEYWORDS):   
     # Collect some examples in information theoretic crypto
    KEYWORDS = [
        "private information retrieval",
        "linear secret sharing",
        "graph secret sharing",
        "monotone span program",
        "CDS",
        "robust secret sharing",
        "verifiable secret sharing",
        "secret sharing",
    ]
    ALL_KEYWORDS += KEYWORDS
    itcr_papers = fetch_arxiv_data(CATEGORIES, 100, logger, DAYS_BACK, KEYWORDS)
    papers += itcr_papers

def collect_csp_papers(papers, ALL_KEYWORDS):
    # Collect some examples on CSP refutation and spectral methods
    KEYWORDS = [
        "matrix concentration",
        "CSP refutation",
        "Kikuchi matrix",
        "spectral method",
        "spectral approach",
        "CSP", 
        "even cover",
        "random walk",
        "mixing time",
    ]
    ALL_KEYWORDS += KEYWORDS
    spectral_papers = fetch_arxiv_data(CATEGORIES, 150, logger, DAYS_BACK, KEYWORDS)
    papers += spectral_papers

def collect_lb_papers(papers, ALL_KEYWORDS):
    # Collect some examples on lower bounds
    KEYWORDS = [
        "rank method",
        "polynomial method",
        "lower bound",
        "entropy method",
        "matrix rigidity",
        "discrepancy",
    ]
    ALL_KEYWORDS += KEYWORDS
    lb_papers = fetch_arxiv_data(CATEGORIES, 75, logger, DAYS_BACK, KEYWORDS)
    papers += lb_papers

def dedup_papers(papers):
    # Deuplicate papers
    already_seen = set()
    deduped = []
    for paper in papers:
        pid = paper["id"].partition("v")[0]
        if pid not in already_seen:
            already_seen.add(pid)
            deduped.append(paper)
    return deduped

def build_dataset(papers, ALL_KEYWORDS):
    collect_coding_papers(papers, ALL_KEYWORDS)
    collect_csp_papers(papers, ALL_KEYWORDS)
    collect_itcr_papers(papers, ALL_KEYWORDS)
    collect_lb_papers(papers, ALL_KEYWORDS)
    collect_locality_papers(papers, ALL_KEYWORDS)
    papers[:] = dedup_papers(papers)

build_dataset(papers, ALL_KEYWORDS)
# Computer some statistics about the data set
num_in_category = {"cs.DS": 0, "cs.IT": 0, "cs.CC": 0, "math.CO": 0, "cs.CR": 0}
num_per_keyword = {kw: 0 for kw in ALL_KEYWORDS}

for paper in papers:
    num_in_category[paper['arxiv_category']] += 1
    for kw in ALL_KEYWORDS: 
        if kw in paper['abstract'] or kw in paper['title']:
            num_per_keyword[kw] += 1


print(f"Number of papers retrieved: {len(papers)}")
print(f"Publication date of oldest paper: {papers[-1]['published']}")
pp(num_in_category)
pp(num_per_keyword)


INFO:root:Collected 0 papers so far. Expect 0.06 more batches.
  9%|▊         | 87/1000 [00:08<01:26, 10.50it/s]
INFO:root:Collected 0 papers so far. Expect 0.12 more batches.
 26%|██▌       | 255/1000 [00:16<00:47, 15.61it/s]
INFO:root:Collected 0 papers so far. Expect 0.08 more batches.
 16%|█▋        | 138/844 [00:08<00:45, 15.43it/s]
INFO:root:Collected 0 papers so far. Expect 0.06 more batches.
  9%|▉         | 91/1000 [00:06<01:03, 14.27it/s]
INFO:root:Collected 0 papers so far. Expect 0.12 more batches.
 31%|███       | 176/576 [00:13<00:31, 12.80it/s]


Number of papers retrieved: 501
Publication date of oldest paper: 2023-01-31T00:57:04
{'cs.DS': 95, 'cs.IT': 235, 'cs.CC': 67, 'math.CO': 104, 'cs.CR': 0}
{'erasure coding': 4,
 'list decoding': 25,
 'list recovery': 6,
 'random code': 11,
 'matching vector': 1,
 'matrix concentration': 4,
 'CSP refutation': 1,
 'Kikuchi matrix': 2,
 'spectral method': 12,
 'spectral approach': 4,
 'CSP': 64,
 'even cover': 3,
 'random walk': 47,
 'mixing time': 20,
 'private information retrieval': 92,
 'linear secret sharing': 0,
 'graph secret sharing': 0,
 'monotone span program': 1,
 'CDS': 2,
 'robust secret sharing': 0,
 'verifiable secret sharing': 0,
 'secret sharing': 28,
 'rank method': 0,
 'polynomial method': 2,
 'lower bound': 151,
 'entropy method': 0,
 'matrix rigidity': 0,
 'discrepancy': 3,
 'locally decodable': 16,
 'locally correctable': 9,
 'locally testable': 13,
 'smooth code': 0,
 'locally recoverable': 30}


In [ ]:
from copy import deepcopy

from paper_clustering.regex_labels import get_enrichment_text


def build_enriched_corpus(papers):
    """Add introduction enrichment and retain one matched evaluation corpus."""
    enriched_papers = []
    for paper in tqdm(papers, desc="Extracting enrichment"):
        enrichment = get_enrichment_text(paper.get("introduction", ""))
        if not enrichment:
            continue
        enriched_paper = deepcopy(paper)
        enriched_paper["enrichment"] = enrichment
        enriched_papers.append(enriched_paper)

    print(f"Matched evaluation corpus: {len(enriched_papers)} papers")
    return enriched_papers


evaluation_papers = build_enriched_corpus(papers)


Extracting enrichment: 100%|██████████| 501/501 [00:49<00:00, 10.18it/s]

Matched evaluation corpus: 501 papers


In [ ]:
import numpy as np
import torch
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer


TEXT_SCOPES = {
    "without_enrichment": ("title", "abstract"),
    "with_enrichment": ("title", "abstract", "enrichment"),
}

EMBEDDING_TEXT_SCOPE = "with_enrichment"

def paper_text(paper, text_scope=EMBEDDING_TEXT_SCOPE):
    """Build the text representation used for one embedding experiment."""
    if text_scope not in TEXT_SCOPES:
        raise ValueError(f"Unknown text scope {text_scope!r}; choose from {tuple(TEXT_SCOPES)}")
    return "\n\n".join(paper[field] for field in TEXT_SCOPES[text_scope])


class BM25Index:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=True)
        self.term_counts = self.vectorizer.fit_transform(documents).tocsr()
        self.k1 = k1
        self.b = b
        self.document_lengths = np.asarray(self.term_counts.sum(axis=1)).ravel()
        self.average_length = self.document_lengths.mean()
        document_frequency = np.asarray((self.term_counts > 0).sum(axis=0)).ravel()
        num_documents = self.term_counts.shape[0]
        self.idf = np.log(1 + (num_documents - document_frequency + 0.5) / (document_frequency + 0.5))

    def score(self, query):
        query_terms = self.vectorizer.transform([query]).indices
        if not len(query_terms):
            return np.zeros(self.term_counts.shape[0])

        frequencies = self.term_counts[:, query_terms].toarray()
        length_penalty = self.k1 * (1 - self.b + self.b * self.document_lengths / self.average_length)
        return (
            self.idf[query_terms]
            * (frequencies * (self.k1 + 1) / (frequencies + length_penalty[:, None]))
        ).sum(axis=1)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)
_nomic_model = None


def embed_nomic(documents, batch_size=2):
    global _nomic_model
    if _nomic_model is None:
        _nomic_model = SentenceTransformer(
            "nomic-ai/nomic-embed-text-v1.5",
            trust_remote_code=True,
            device=DEVICE,
        )
        _nomic_model.max_seq_length = 2048

    clustering_documents = [f"clustering: {document}" for document in documents]
    return _nomic_model.encode(
        clustering_documents,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

base_documents = [paper_text(paper, "without_enrichment") for paper in evaluation_papers]
enriched_documents = [paper_text(paper, "with_enrichment") for paper in evaluation_papers]

evaluation_indexes = {
    "nomic_with_enrichment": embed_nomic(enriched_documents, batch_size=4),
    "bm25_with_enrichment": BM25Index(enriched_documents),
    "bm25_without_enrichment": BM25Index(base_documents),
}
print(f"Built all three indexes over {len(evaluation_papers)} matched papers")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nomic-ai/nomic-embed-text-v1.5/e9b6763023c676ca8431644204f50c2b100d9aab/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


cuda


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nomic-ai/nomic-embed-text-v1.5/e9b6763023c676ca8431644204f50c2b100d9aab/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from nomic-ai/nomic-embed-text-v1.5.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nomic-ai/nomic-embed-text-v1.5/e9b6763023c676ca8431644204f50c2b100d9aab/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nomic-ai/nomic-embed-text-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nomic-ai/nomic-embed-text-v1.5/e9b6763023c676ca8431644204f50c2b100d9aab/README.md "HTTP/1.1 200 OK"
INF

Batches:   0%|          | 0/126 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Built all three indexes over 501 matched papers


## Compare enrichment indexes

The cells below compare the top neighbours returned by **Nomic with enrichment**, **BM25 with enrichment**, and **BM25 without enrichment**. Scores should only be compared within a method: cosine similarity and BM25 scores are on different scales.

In [5]:
from itertools import combinations
from IPython.display import display


CONFIGURATIONS = (
    "nomic_with_enrichment",
    "bm25_with_enrichment",
    "bm25_without_enrichment",
)


def paper_position(paper_id_or_title):
    """Resolve an arXiv ID, exact title, or unique title fragment."""
    exact = [
        index
        for index, paper in enumerate(evaluation_papers)
        if paper_id_or_title in (paper["id"], paper["title"])
    ]
    if len(exact) == 1:
        return exact[0]

    matches = [
        index
        for index, paper in enumerate(evaluation_papers)
        if paper_id_or_title.lower() in paper["title"].lower()
    ]
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise KeyError(f"No paper matched {paper_id_or_title!r}")
    raise ValueError(
        f"Multiple papers matched {paper_id_or_title!r}: "
        + repr([evaluation_papers[index]["title"] for index in matches[:10]])
    )


def configuration_scores(configuration, query_position):
    """Score every paper using the matching query representation."""
    index = evaluation_indexes[configuration]
    if configuration == "nomic_with_enrichment":
        return index @ index[query_position]
    if configuration == "bm25_with_enrichment":
        return index.score(enriched_documents[query_position])
    return index.score(base_documents[query_position])


def ranked_positions(configuration, query_position, k=10):
    scores = configuration_scores(configuration, query_position)
    positions = [
        position
        for position in np.argsort(-scores)
        if position != query_position
    ][:k]
    return positions, scores


def compare_nearest_neighbours(paper_id_or_title, k=10):
    """Display the three rankings side by side for one query paper."""
    query_position = paper_position(paper_id_or_title)
    query = evaluation_papers[query_position]
    columns = {"rank": list(range(1, k + 1))}

    for configuration in CONFIGURATIONS:
        positions, scores = ranked_positions(configuration, query_position, k)
        columns[configuration] = [
            f"{evaluation_papers[position]['title']} "
            f"[{float(scores[position]):.4f}]"
            for position in positions
        ]

    print(f"Query: {query['title']} ({query['id']})")
    comparison = pd.DataFrame(columns).set_index("rank")
    display(comparison)
    return comparison


In [6]:
# Reuse the two case studies from the baseline embedding evaluation.
CASE_STUDIES = [
    "Exponential Lower Bounds for Smooth 3-LCCs",
    "Efficient Catalytic Graph Algorithms",
]

for case_study in CASE_STUDIES:
    try:
        compare_nearest_neighbours(case_study, k=10)
    except KeyError as error:
        print(error)


def mean_top_k_overlap(k=10):
    """Measure how often each pair of methods returns the same neighbours."""
    totals = {pair: [] for pair in combinations(CONFIGURATIONS, 2)}
    for query_position in tqdm(range(len(evaluation_papers)), desc="Comparing rankings"):
        neighbours = {
            configuration: set(ranked_positions(configuration, query_position, k)[0])
            for configuration in CONFIGURATIONS
        }
        for pair in totals:
            totals[pair].append(len(neighbours[pair[0]] & neighbours[pair[1]]) / k)

    return pd.DataFrame(
        [
            {"configuration_a": left, "configuration_b": right, "mean_overlap@k": np.mean(values)}
            for (left, right), values in totals.items()
        ]
    )


display(mean_top_k_overlap(k=10))

Query: Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs (2404.06513v2)


,nomic_with_enrichment,bm25_with_enrichment,bm25_without_enrichment
rank,,,
1,An Exponential Lower Bound for Linear 3-Query ...,An Exponential Lower Bound for Linear 3-Query ...,An Exponential Lower Bound for Linear 3-Query ...
2,Near-Tight Bounds for 3-Query Locally Correcta...,A Near-Cubic Lower Bound for 3-Query Locally D...,Near-Tight Bounds for 3-Query Locally Correcta...
3,A Near-Cubic Lower Bound for 3-Query Locally D...,A $k^{\frac{q}{q-2}}$ Lower Bound for Odd Quer...,A $k^{\frac{q}{q-2}}$ Lower Bound for Odd Quer...
4,Exponential Lower Bounds for 2-query Relaxed L...,Relaxed vs. Full Local Decodability with Few Q...,A Near-Cubic Lower Bound for 3-Query Locally D...
5,Improved Lower Bounds for all Odd-Query Locall...,Improved Lower Bounds for all Odd-Query Locall...,Improved Lower Bounds for all Odd-Query Locall...
6,Good Locally Testable Codes with Small Alphabe...,Local Correction of Linear Functions over the ...,"Small Even Covers, Locally Decodable Codes and..."
7,Relaxed vs. Full Local Decodability with Few Q...,Near-Tight Bounds for 3-Query Locally Correcta...,Sensitivity Lower Bounds via Locally Testable ...
8,3-Query RLDCs are Strictly Stronger than 3-Que...,"Small Even Covers, Locally Decodable Codes and...",Exponential Lower Bounds for 2-query Relaxed L...
9,A $k^{\frac{q}{q-2}}$ Lower Bound for Odd Quer...,Sensitivity Lower Bounds via Locally Testable ...,Relaxed vs. Full Local Decodability with Few Q...


Query: Efficient Catalytic Graph Algorithms (2509.06209v1)


,nomic_with_enrichment,bm25_with_enrichment,bm25_without_enrichment
rank,,,
1,Catalytic Tree Evaluation From Matching Vector...,DNF Learning via Locally Mixing Random Walks [...,Estimating Random-Walk Probabilities in Direct...
2,Streaming with Catalytic Memory [0.8263],Note on the trace of random walks on pseudoran...,Improved Algorithms for Effective Resistance C...
3,An algebraic-combinatorial framework for findi...,Catalytic Tree Evaluation From Matching Vector...,Catalytic Tree Evaluation From Matching Vector...
4,Randomization Helps in Online Graph Exploratio...,Streaming with Catalytic Memory [102.6876],Randomization Helps in Online Graph Exploratio...
5,Estimating Hitting Times Locally At Scale [0.8...,A Dichotomy Theorem for Multi-Pass Streaming C...,Complexity of the Graph Homomorphism Problem w...
6,Estimating Random-Walk Probabilities in Direct...,Complexity of the Graph Homomorphism Problem w...,Estimating Hitting Times Locally At Scale [44....
7,Scalable and Provable Kemeny Constant Computat...,Sensitivity Lower Bounds via Locally Testable ...,Streaming with Catalytic Memory [44.2463]
8,Graph-based Nearest Neighbors with Dynamic Upd...,Deterministic Dynamic Maximal Matching in Subl...,Triangular cutoff threshold for the inversion ...
9,Note on the trace of random walks on pseudoran...,The communication complexity of distributed es...,Spectral Clustering in Birthday Paradox Time [...


Comparing rankings: 100%|██████████| 501/501 [00:01<00:00, 260.47it/s]


,configuration_a,configuration_b,mean_overlap@k
0,nomic_with_enrichment,bm25_with_enrichment,0.351497
1,nomic_with_enrichment,bm25_without_enrichment,0.361477
2,bm25_with_enrichment,bm25_without_enrichment,0.568862
